In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

ROI = "HIP"
CHEMS = None

df = pd.read_parquet(f"{SLIM_DIR}/{ROI}_slim.parquet")
if CHEMS:
    df = df[df["chem"].isin(CHEMS)]
genes = [g for g in GRMS if g in df.columns]
tag = ROI + ("_" + "-".join(c.replace("WMB-10X", "") for c in CHEMS) if CHEMS else "")
rname = f"{NAMES.get(ROI, ROI)} ({ROI})"
chem_str = ", ".join(sorted(c.replace("WMB-10X", "10x ").replace("Multi", "Multiome") for c in df["chem"].astype(str).unique()))
print(ROI, len(df), "cells;", df["subclass"].nunique(), "subclasses; chem:", df["chem"].value_counts().to_dict())

rng = np.random.default_rng(0)
info = df.groupby("subclass").agg(n=("cluster", "size"), cls=("class", "first"))
info = info[info["n"] >= MIN_CELLS].sort_values(["cls"]).reset_index()
info = info.sort_values(["cls", "subclass"]).reset_index(drop=True)
order = info["subclass"].tolist()
groups = {s: g for s, g in df.groupby("subclass")}
fig, axes = plt.subplots(len(genes), 1, figsize=(max(8, 0.3 * len(order)), 2.1 * len(genes)), sharex=True)
axes = np.atleast_1d(axes)
for ax, g in zip(axes, genes):
    for i, s in enumerate(order):
        v = groups[s][g].to_numpy()
        colr = class_color(info.loc[i, "cls"])
        if np.ptp(v) > 0:
            parts = ax.violinplot(v, positions=[i], widths=0.85, showextrema=False)
            for b in parts["bodies"]:
                b.set_facecolor(colr)
                b.set_alpha(0.8)
                b.set_edgecolor("none")
        pts = rng.choice(v, size=min(40, len(v)), replace=False)
        dark = tuple(c_ * 0.5 for c_ in colr[:3])
        ax.scatter(i + rng.uniform(-0.3, 0.3, len(pts)), pts, s=1.6, color=dark, alpha=0.6, linewidths=0, rasterized=True)
        ax.plot([i - 0.3, i + 0.3], [np.median(v)] * 2, color=dark, lw=1)
    ax.set_ylabel(g + "\nlog2(CPM+1)", fontsize=8)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
axes[-1].set_xticks(range(len(order)))
axes[-1].set_xticklabels(order, rotation=90, fontsize=6)
for lab, s in zip(axes[-1].get_xticklabels(), order):
    lab.set_color(class_color(info.loc[order.index(s), "cls"]))
axes[0].set_title(f"{rname}: Grm expression per cell by subclass ({len(df):,} cells; {chem_str})\nlog2(CPM+1); violin = distribution, dots = up to 40 random cells (darker shade of the class color), bar = median; subclasses with >= {MIN_CELLS} cells; color = class (as UMAP)", fontsize=9)
cls_in = sorted(info["cls"].unique())
axes[0].legend(handles=[plt.Line2D([], [], marker="s", ls="", color=class_color(c), label=c, markersize=6) for c in cls_in], fontsize=6, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), title="class")
fig.savefig(f"{FIG_DIR}/{tag}_fig2_violins.png", dpi=300, bbox_inches="tight")
plt.show()